# 1. EMR 6.x Hands-On: S3, HDFS, and Hive

This lab performs one complete storage workflow on an Amazon EMR 6.x cluster:

1. Create and upload a CSV file to Amazon S3
2. Access S3 through EMRFS
3. Copy data from S3 to HDFS and from HDFS to S3
4. Create a Hive external table over S3 data
5. Create a Hive managed table in the configured warehouse
6. Verify table locations and `DROP TABLE` behavior

Run the commands on the EMR primary node or in a notebook attached to the cluster. Use an EMR 6.x release with Hadoop and Hive installed.

# 2. Set the Lab Values

Replace the bucket name below. The bucket must already exist and should be in the same Region as the cluster. Bucket names are globally unique.

The EMR EC2 instance profile needs permission for the required S3 actions on the bucket and objects, including listing, reading, writing, and deleting the temporary verification object if cleanup is performed.

Every shell cell sets its own variables so it can run independently.

In [ ]:
%%bash
BUCKET=replace-with-your-unique-bucket-name
PREFIX=d264-emr-hive-lab
REGION=$(aws configure get region)
echo "Region: ${REGION}"
echo "S3 base: s3://${BUCKET}/${PREFIX}/"
aws sts get-caller-identity --query '{Account:Account,Arn:Arn}'

# 3. Confirm EMR 6.x and EMRFS

For EMR 6.x, `s3://` is handled by **EMRFS**, the Amazon EMR connector that translates Hadoop filesystem operations into S3 API requests. This differs from EMR 7.10.0 and later, where S3A became the default connector for S3 schemes.

The following commands confirm the cluster release and show the class configured for `fs.s3.impl`. Expected on EMR 6.x: an EMRFS implementation such as `com.amazon.ws.emr.hadoop.fs.EmrFileSystem`.

In [ ]:
%%bash
echo "EMR release:"
cat /emr/instance-controller/lib/info/extraInstanceData.json 2>/dev/null | grep -o '"releaseLabel"[^,]*' || true
echo
echo "S3 filesystem implementation:"
hadoop getconf -confKey fs.s3.impl
echo
echo "HDFS default filesystem:"
hadoop getconf -confKey fs.defaultFS

# 4. Create a Small Local Dataset

The file starts on the primary node's local filesystem. Local storage, HDFS, and S3 are three different namespaces. Always identify a path by its scheme and purpose.

In [ ]:
%%bash
printf 'order_id,customer,amount,order_date\n1,Asha,1200.50,2026-08-18\n2,Ravi,850.00,2026-08-19\n3,Meera,640.25,2026-08-20\n' > /tmp/d264_orders.csv
echo "Local file details:"
ls -lh /tmp/d264_orders.csv
echo
cat /tmp/d264_orders.csv

# 5. Upload the Local File to S3

`aws s3 cp` uses the AWS CLI. It uploads from the local filesystem to S3; it does not copy directly from HDFS.

The destination prefix is dedicated to the external table. Keep unrelated files out of a Hive table location because Hive attempts to interpret files under that location as table data.

In [ ]:
%%bash
BUCKET=replace-with-your-unique-bucket-name
PREFIX=d264-emr-hive-lab
aws s3 cp /tmp/d264_orders.csv "s3://${BUCKET}/${PREFIX}/external_orders/orders.csv"
aws s3 ls "s3://${BUCKET}/${PREFIX}/external_orders/"

# 6. Access S3 Through Hadoop on EMR 6.x

`hadoop fs` selects a filesystem from the URI scheme. With `s3://` on EMR 6.x, Hadoop delegates the operation to EMRFS. EMRFS makes S3 API calls using permissions available to the cluster, normally through the EC2 instance profile unless EMRFS role mapping is configured.

The bytes remain in S3. Listing or reading them does not first place a permanent copy in HDFS.

In [ ]:
%%bash
BUCKET=replace-with-your-unique-bucket-name
PREFIX=d264-emr-hive-lab
hadoop fs -ls "s3://${BUCKET}/${PREFIX}/external_orders/"
echo
hadoop fs -cat "s3://${BUCKET}/${PREFIX}/external_orders/orders.csv"

# 7. Copy from S3 to HDFS

Use Hadoop filesystem commands when either side is HDFS. The source is read through EMRFS and the destination is written as HDFS blocks on the cluster's core nodes.

`-cp` copies between Hadoop-compatible filesystems. `-copyToLocal` is different: it copies to the local filesystem of the machine running the command.

In [ ]:
%%bash
BUCKET=replace-with-your-unique-bucket-name
PREFIX=d264-emr-hive-lab
HDFS_DIR=hdfs:///user/hadoop/d264/input
hadoop fs -mkdir -p "${HDFS_DIR}"
hadoop fs -cp "s3://${BUCKET}/${PREFIX}/external_orders/orders.csv" "${HDFS_DIR}/orders.csv"
hadoop fs -ls "${HDFS_DIR}"
hadoop fs -cat "${HDFS_DIR}/orders.csv"

# 8. Copy from HDFS to S3

The reverse operation reads HDFS blocks and writes an S3 object through EMRFS. Use a separate destination prefix so verification is unambiguous.

For very large transfers or recursive production copies, S3DistCp can add distributed execution, retries, grouping, and compression options. The simple `hadoop fs -cp` command is sufficient for this small file.

In [ ]:
%%bash
BUCKET=replace-with-your-unique-bucket-name
PREFIX=d264-emr-hive-lab
hadoop fs -cp hdfs:///user/hadoop/d264/input/orders.csv "s3://${BUCKET}/${PREFIX}/hdfs_export/orders.csv"
hadoop fs -ls "s3://${BUCKET}/${PREFIX}/hdfs_export/"
echo
hadoop fs -cat "s3://${BUCKET}/${PREFIX}/hdfs_export/orders.csv"

# 9. Create a Hive External Table on S3

An external table stores table metadata in the Hive metastore while its `LOCATION` points to separately managed data. Here the location is an S3 prefix, and Hive reads it through EMRFS.

The CSV has a header, so the table property skips the first line. The table location is a subpath, not the bucket root.

In [ ]:
%%bash
BUCKET=replace-with-your-unique-bucket-name
PREFIX=d264-emr-hive-lab
hive --hiveconf LAB_BUCKET="${BUCKET}" --hiveconf LAB_PREFIX="${PREFIX}" -e "
CREATE DATABASE IF NOT EXISTS d264_lab;
USE d264_lab;
DROP TABLE IF EXISTS orders_external_s3;
CREATE EXTERNAL TABLE orders_external_s3 (
  order_id BIGINT,
  customer STRING,
  amount DECIMAL(10,2),
  order_date DATE
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
STORED AS TEXTFILE
LOCATION 's3://\${hiveconf:LAB_BUCKET}/\${hiveconf:LAB_PREFIX}/external_orders/'
TBLPROPERTIES ('skip.header.line.count'='1');
SELECT * FROM orders_external_s3 ORDER BY order_id;
DESCRIBE FORMATTED orders_external_s3;
"

# 10. Verify External-Table Ownership

`DROP TABLE` removes the external table definition from the metastore. It does not normally delete the S3 objects owned outside Hive.

Run the drop, confirm the table is gone, and then list the S3 prefix. The CSV should still exist.

In [ ]:
%%bash
BUCKET=replace-with-your-unique-bucket-name
PREFIX=d264-emr-hive-lab
hive -e "USE d264_lab; DROP TABLE orders_external_s3; SHOW TABLES;"
echo "S3 data after dropping the external table:"
hadoop fs -ls "s3://${BUCKET}/${PREFIX}/external_orders/"

# 11. Create a Hive Managed Table

A managed table has no explicit `EXTERNAL` keyword and, in this example, no explicit `LOCATION`. Hive chooses a directory under the active warehouse location and treats the table data as Hive-owned.

On a standard EMR 6.x configuration, the warehouse commonly resolves to HDFS, often under `/user/hive/warehouse`. Do not rely on memory: inspect `hive.metastore.warehouse.dir` and `DESCRIBE FORMATTED`. A customized cluster can point the warehouse elsewhere.

The `INSERT` reads the CSV from S3 through the external table and writes managed-table files to the configured warehouse.

In [ ]:
%%bash
BUCKET=replace-with-your-unique-bucket-name
PREFIX=d264-emr-hive-lab
hive --hiveconf LAB_BUCKET="${BUCKET}" --hiveconf LAB_PREFIX="${PREFIX}" -e "
SET hive.metastore.warehouse.dir;
USE d264_lab;
CREATE EXTERNAL TABLE IF NOT EXISTS orders_external_s3 (
  order_id BIGINT, customer STRING, amount DECIMAL(10,2), order_date DATE
)
ROW FORMAT DELIMITED FIELDS TERMINATED BY ','
STORED AS TEXTFILE
LOCATION 's3://\${hiveconf:LAB_BUCKET}/\${hiveconf:LAB_PREFIX}/external_orders/'
TBLPROPERTIES ('skip.header.line.count'='1');
DROP TABLE IF EXISTS orders_managed;
CREATE TABLE orders_managed (
  order_id BIGINT, customer STRING, amount DECIMAL(10,2), order_date DATE
) STORED AS ORC;
INSERT INTO orders_managed SELECT * FROM orders_external_s3;
SELECT * FROM orders_managed ORDER BY order_id;
DESCRIBE FORMATTED orders_managed;
"

# 12. Locate the Managed Data

Read the `Location:` value printed by `DESCRIBE FORMATTED`. If it begins with `hdfs://` or has an HDFS path, list it with `hadoop fs`.

For the default database, a common path is `/user/hive/warehouse/orders_managed`. Because this lab uses database `d264_lab`, a common database directory is `/user/hive/warehouse/d264_lab.db/orders_managed`. Confirm the actual value before running a path-specific command.

In [ ]:
%%bash
echo "Configured Hive warehouse:"
hive --silent -e 'SET hive.metastore.warehouse.dir;'
echo
echo "Managed table location:"
hive --silent -e "USE d264_lab; DESCRIBE FORMATTED orders_managed;" | grep -i -m1 'Location'
echo
echo "Common default HDFS location, if present:"
hadoop fs -ls hdfs:///user/hive/warehouse/d264_lab.db/orders_managed 2>/dev/null || echo 'Use the Location value displayed above.'

# 13. Verify Managed-Table Ownership

For a managed table, Hive controls both metadata and table data. Under normal Hive managed-table behavior, `DROP TABLE` removes the metastore definition and deletes the table directory.

First record the exact location from the preceding step. After dropping the table, verify that location. The external-table CSV in S3 remains independent.

Do not run this pattern on valuable data until ownership and retention behavior are understood.

In [ ]:
%%bash
hive -e "USE d264_lab; DROP TABLE orders_managed;"
echo "Check the common default managed-table path:"
hadoop fs -test -e hdfs:///user/hive/warehouse/d264_lab.db/orders_managed
if [ $? -eq 0 ]; then echo 'Path still exists: inspect the actual table configuration.'; else echo 'Managed-table path was removed.'; fi

# 14. Managed and External Tables: Exact Distinction

| Question | External table on S3 | Managed table in default EMR 6.x warehouse |
|---|---|---|
| Who owns the data lifecycle? | External process or data platform | Hive |
| SQL marker | `CREATE EXTERNAL TABLE` | `CREATE TABLE` |
| Location in this lab | Explicit `s3://...` prefix | Chosen under `hive.metastore.warehouse.dir` |
| Read path | Hive → EMRFS → S3 API → objects | Hive → HDFS client → NameNode/DataNodes |
| Normal drop behavior | Metadata removed; objects remain | Metadata and table directory removed |
| Survives cluster termination | Yes, because objects are in S3 | No when warehouse data is in cluster HDFS |

`MANAGED` versus `EXTERNAL` defines ownership. `S3` versus `HDFS` defines storage location. They are related in this lab but are not synonyms. Always verify the `Location` field.

# 15. Cleanup and Final Verification

The cleanup below drops the recreated external-table metadata, removes the temporary HDFS directory, and removes only the dedicated S3 lab prefix. Review the bucket and prefix before execution.

In [ ]:
%%bash
BUCKET=replace-with-your-unique-bucket-name
PREFIX=d264-emr-hive-lab
hive -e "USE d264_lab; DROP TABLE IF EXISTS orders_external_s3; DROP DATABASE IF EXISTS d264_lab;"
hadoop fs -rm -r -skipTrash hdfs:///user/hadoop/d264
echo "Review target before removing S3 objects: s3://${BUCKET}/${PREFIX}/"
aws s3 rm "s3://${BUCKET}/${PREFIX}/" --recursive
echo "Cleanup complete."

# 16. Operational Summary

- On EMR 6.x, `s3://` access from Hadoop and Hive uses **EMRFS** by default.
- `aws s3 cp` transfers between the local filesystem and S3.
- `hadoop fs -cp` transfers between Hadoop-supported filesystems, including HDFS and S3 through EMRFS.
- An S3 external table keeps metadata separate from durable objects.
- A managed table gives Hive ownership of the data at the configured warehouse location.
- Inspect `hive.metastore.warehouse.dir` and `DESCRIBE FORMATTED`; do not guess the physical location.
- Preserve important outputs in S3 before an EMR cluster terminates.

References: [EMR filesystems](https://docs.aws.amazon.com/emr/latest/ManagementGuide/emr-plan-file-systems.html), [EMRFS](https://docs.aws.amazon.com/emr/latest/ReleaseGuide/emr-fs.html), [EMRFS IAM roles](https://docs.aws.amazon.com/emr/latest/ManagementGuide/emr-emrfs-iam-roles.html), and [Hive differences on EMR](https://docs.aws.amazon.com/emr/latest/ReleaseGuide/emr-hive-differences.html).